# 🏛️ PolyScribe AI - Complete Model Training & Translation Pipeline
### End-to-End Deep Learning Training on Google Colab (Free T4 GPU)

This notebook trains and benchmarks all models for **PolyScribe AI**:
1. **Script Classifier CNN (6-class)**: Identifies `Kannada`, `Tamil`, `Telugu`, `Arabic`, `German / Latin`, and `Devanagari`.
2. **Shared Indic-CRNN + CTC**: Deep Bidirectional GRU model trained to transcribe handwritten Kannada, Tamil, and Telugu.
3. **Latin / German Recognizer**: Fine-tuned TrOCR & EasyOCR handwriting models.
4. **Meta NLLB-200 Translation Engine**: Translates all 5 languages into fluent English (`eng_Latn`).
5. **Export to Google Drive**: Automatically packages weights for deployment.


## 1. Environment & GPU Verification

In [ ]:
# Check NVIDIA GPU acceleration
!nvidia-smi

import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. Clone Repository & Install Dependencies

In [ ]:
!git clone https://github.com/Sindhunavarathna/major_project.git /content/major_project
%cd /content/major_project

# Install required dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers sentencepiece easyocr opencv-python Pillow numpy scikit-learn matplotlib albumentations
!apt-get install -y -q fonts-dejavu-core fonts-noto-core fonts-noto-cjk fonts-noto-ui-core fonts-noto-extra

## 3. Train Script Classifier CNN (Kannada, Tamil, Telugu, Arabic, Latin, Devanagari)
Trains the 6-class visual CNN on authentic augmented manuscript patches.

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from src.script_classifier.dataset import get_train_val_test_dataloaders
from src.script_classifier.model import ScriptClassifierCNN, CLASS_NAMES

print(f'Training Script Classifier across {len(CLASS_NAMES)} classes: {CLASS_NAMES}')

# Generate balanced dataset on Colab
train_loader, val_loader, test_loader, splits = get_train_val_test_dataloaders(
    batch_size=32,
    samples_per_class=350,
    seed=42
)
print(f'Dataset generated: {splits}')

# Initialize CNN on GPU
model = ScriptClassifierCNN(num_classes=len(CLASS_NAMES), in_channels=1).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

os.makedirs('models/script_classifier', exist_ok=True)
best_acc = 0.0
save_path = 'models/script_classifier/best_script_classifier.pth'

print('Starting Training on GPU...')
for epoch in range(1, 21):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        preds = out.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)
    scheduler.step()
    train_acc = correct / total

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            preds = out.argmax(dim=1)
            val_correct += (preds == y).sum().item()
            val_total += x.size(0)
    val_acc = val_correct / val_total
    print(f'Epoch {epoch:02d}/20 - Train Loss: {total_loss/total:.4f} | Train Acc: {train_acc*100:.1f}% | Val Acc: {val_acc*100:.1f}%')

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'classes': CLASS_NAMES,
            'val_acc': best_acc
        }, save_path)
        print(f'  --> Saved new best checkpoint with Val Acc: {best_acc*100:.2f}%')

## 4. Train Shared Indic-CRNN + CTC Engine
Trains the deep Bidirectional GRU model on Indic text lines (Kannada, Tamil, Telugu).

In [ ]:
from src.indic_ocr.model import IndicCRNN
from src.indic_ocr.vocabulary import IndicVocabulary
from src.indic_ocr.dataset import get_indic_dataloaders

vocab = IndicVocabulary()
print(f'Indic Vocabulary size: {len(vocab)}')

indic_model = IndicCRNN(num_classes=len(vocab), hidden_size=256, num_layers=2).to(device)
print(f'Indic-CRNN parameters: {sum(p.numel() for p in indic_model.parameters()):,}')

# Run training script
!python -m src.indic_ocr.train --epochs 15 --batch-size 32 --lr 0.0005 --save-dir models/indic_crnn

## 5. Test Multi-Language Translation with Meta NLLB-200
Benchmarks translation of all 5 target languages (German, Arabic, Kannada, Telugu, Tamil) into English.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = 'facebook/nllb-200-distilled-600M'
print(f'Loading {model_name} onto GPU...')
tokenizer = AutoTokenizer.from_pretrained(model_name)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# Target language test sentences
test_sentences = [
    ('German', 'deu_Latn', 'Die alte Handschrift enthaelt wertvolles Wissen vergangener Jahrhunderte.'),
    ('Arabic', 'arb_Arab', 'العلم نور يهدي الإنسان في ظلمات الجهل والحيرة.'),
    ('Kannada', 'kan_Knda', 'ಕನ್ನಡ ಭಾಷೆಯ ಪ್ರಾಚೀನ ಸಾಹಿತ್ಯವು ಅತ್ಯಂತ ಶ್ರೀಮಂತ ಮತ್ತು ಶ್ರೇಷ್ಠವಾಗಿದೆ.'),
    ('Telugu', 'tel_Telu', 'తెలుగు భాష మరియు సాహిత్యం ఎంతో ప్రాచీనమైనవి మరియు గొప్ప సాంస్కృతిక వారసత్వం కలిగి ఉన్నాయి.'),
    ('Tamil', 'tam_Taml', 'தமிழ் மொழி உலகின் மிக தொன்மையான செம்மொழிகளில் ஒன்றாகும்.')
]

print('\n' + '='*70)
print('TRANSLATION BENCHMARK TO ENGLISH (eng_Latn)')
print('='*70)

eng_token_id = tokenizer.convert_tokens_to_ids('eng_Latn')

for lang_name, lang_code, text in test_sentences:
    tokenizer.src_lang = lang_code
    inputs = tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
        generated_tokens = nllb_model.generate(
            **inputs,
            forced_bos_token_id=eng_token_id,
            max_length=128
        )
    translation = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    print(f'[{lang_name}] Original:    {text}')
    print(f'[{lang_name}] Translation: {translation}')
    print('-'*70)

## 6. End-to-End Inference Verification
Tests sample manuscripts from `sample_data/` through the complete pipeline.

In [ ]:
from src.fast_pipeline import FastPipelineManager

pipeline = FastPipelineManager.get_instance()

samples = [
    ('sample_data/sample_german.png', 'German Note'),
    ('sample_data/sample_arabic.png', 'Arabic Note'),
    ('sample_data/sample_kannada.png', 'Kannada Note'),
    ('sample_data/sample_telugu.png', 'Telugu Note'),
    ('sample_data/sample_tamil.png', 'Tamil Note')
]

for img_path, title in samples:
    if os.path.exists(img_path):
        res = pipeline.run_inference(img_path)
        print(f'\n=== {title} ===')
        print(f'Detected Language: {res.detected_language} (Script: {res.detected_script}, Confidence: {res.confidence*100:.1f}%)')
        print(f'Recognized: {res.recognized_text[:80]}...')
        print(f'Translation: {res.translated_text[:100]}...')
        print(f'Timing: Preprocess={res.timing.preprocess_ms:.1f}ms, Script={res.timing.script_classifier_ms:.1f}ms, OCR={res.timing.ocr_ms:.1f}ms, Translate={res.timing.translation_ms:.1f}ms')

## 7. Export Trained Models to Google Drive
Zip and export your trained checkpoints directly to Google Drive.

In [ ]:
from google.colab import drive
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Archive trained models
shutil.make_archive('/content/polyscribe_models', 'zip', '/content/major_project/models')
shutil.copy('/content/polyscribe_models.zip', '/content/drive/MyDrive/polyscribe_models.zip')
print('Successfully saved trained model bundle to Google Drive: MyDrive/polyscribe_models.zip!')